# FoodSeg103 semantic segmentation training

Clean notebook for preparing FoodSeg103 masks as YOLO segmentation labels and training `yolo26s-sem.pt`.

In [1]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -U ultralytics opencv-python tqdm pyyaml

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import shutil

import cv2
import numpy as np
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

c:\training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.12.0+cu130
cuda: True
device: NVIDIA GeForce RTX 5070 Ti


In [4]:
ROOT = Path(r"C:/training/segmentation_clean")
if not (ROOT / "Images").exists():
    ROOT = Path.cwd()

SRC_IMG_TRAIN = ROOT / "Images" / "img_dir" / "train"
SRC_IMG_VAL = ROOT / "Images" / "img_dir" / "test"
SRC_ANN_TRAIN = ROOT / "Images" / "ann_dir" / "train"
SRC_ANN_VAL = ROOT / "Images" / "ann_dir" / "test"
CATEGORY_FILE = ROOT / "category_id.txt"

YOLO_ROOT = ROOT / "yolo_dataset_clean_sem"
DATA_YAML = ROOT / "food103_clean_sem.yaml"
MODEL_WEIGHTS = ROOT / "yolo26l-sem.pt"
PROJECT_DIR = ROOT / "runs" / "segment"
RUN_NAME = "yolo26l_seg_foodseg103"

for required_path in [SRC_IMG_TRAIN, SRC_IMG_VAL, SRC_ANN_TRAIN, SRC_ANN_VAL, CATEGORY_FILE]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

if not MODEL_WEIGHTS.exists():
    print(f"Weights are not present locally: {MODEL_WEIGHTS}")
    print("Training will still try YOLO('yolo26l-seg.pt'), which may download or fail depending on the environment.")

print("root:", ROOT)
print("dataset:", YOLO_ROOT)
print("yaml:", DATA_YAML)

root: C:\training\segmentation_clean
dataset: C:\training\segmentation_clean\yolo_dataset_clean_sem
yaml: C:\training\segmentation_clean\food103_clean_sem.yaml


In [5]:
def read_foodseg_names(category_file):
    names = {}
    with category_file.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            raw_id, class_name = line.split(maxsplit=1)
            mask_id = int(raw_id)
            if mask_id == 0:
                continue

            names[mask_id - 1] = class_name.strip()

    expected = set(range(len(names)))
    actual = set(names)
    if actual != expected:
        missing = sorted(expected - actual)
        extra = sorted(actual - expected)
        raise ValueError(f"Invalid class ids. Missing={missing}, extra={extra}")

    return names


names = read_foodseg_names(CATEGORY_FILE)
print(f"classes: {len(names)}")
print(dict(list(names.items())[:5]))

classes: 103
{0: 'candy', 1: 'egg tart', 2: 'french fries', 3: 'chocolate', 4: 'biscuit'}


In [6]:
REBUILD_DATASET = True

if REBUILD_DATASET and YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

for split in ["train", "val"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

print("dataset directories ready")

dataset directories ready


In [7]:
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
MIN_CONTOUR_AREA = 4.0


def find_image(mask_path, image_dir):
    stem = mask_path.stem
    for extension in IMAGE_EXTENSIONS:
        candidate = image_dir / f"{stem}{extension}"
        if candidate.exists():
            return candidate

    matches = sorted(image_dir.glob(f"{stem}.*"))
    return matches[0] if matches else None


def contour_to_yolo_polygon(contour, width, height):
    if cv2.contourArea(contour) < MIN_CONTOUR_AREA:
        return None

    epsilon = 0.001 * cv2.arcLength(contour, closed=True)
    polygon = cv2.approxPolyDP(contour, epsilon, closed=True)
    if len(polygon) < 3:
        return None

    points = polygon.reshape(-1, 2).astype(np.float32)
    points[:, 0] = np.clip(points[:, 0] / width, 0.0, 1.0)
    points[:, 1] = np.clip(points[:, 1] / height, 0.0, 1.0)

    return " ".join(f"{x:.6f} {y:.6f}" for x, y in points)


def convert_split(image_dir, mask_dir, image_dest, label_dest):
    stats = {"masks": 0, "images": 0, "labels": 0, "objects": 0, "missing_images": 0}
    mask_paths = sorted(mask_dir.glob("*.png"))

    for mask_path in tqdm(mask_paths, desc=mask_dir.name):
        stats["masks"] += 1
        image_path = find_image(mask_path, image_dir)
        if image_path is None:
            stats["missing_images"] += 1
            continue

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue

        height, width = mask.shape[:2]
        label_lines = []
        for mask_id in np.unique(mask):
            if mask_id == 0:
                continue

            class_id = int(mask_id) - 1
            if class_id not in names:
                continue

            binary_mask = (mask == mask_id).astype(np.uint8) * 255
            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for contour in contours:
                polygon = contour_to_yolo_polygon(contour, width, height)
                if polygon is None:
                    continue

                label_lines.append(f"{class_id} {polygon}")

        if not label_lines:
            continue

        shutil.copy2(image_path, image_dest / image_path.name)
        (label_dest / f"{mask_path.stem}.txt").write_text("\n".join(label_lines) + "\n", encoding="utf-8")
        stats["images"] += 1
        stats["labels"] += 1
        stats["objects"] += len(label_lines)

    return stats

In [8]:
train_stats = convert_split(
    SRC_IMG_TRAIN,
    SRC_ANN_TRAIN,
    YOLO_ROOT / "images" / "train",
    YOLO_ROOT / "labels" / "train",
)
val_stats = convert_split(
    SRC_IMG_VAL,
    SRC_ANN_VAL,
    YOLO_ROOT / "images" / "val",
    YOLO_ROOT / "labels" / "val",
)

print("train:", train_stats)
print("val:", val_stats)

test: 100%|██████████| 2121/2121 [00:15<00:00, 140.10it/s]

train: {'masks': 4944, 'images': 4944, 'labels': 4944, 'objects': 28033, 'missing_images': 0}
val: {'masks': 2121, 'images': 2121, 'labels': 2121, 'objects': 11937, 'missing_images': 0}


In [9]:
data_yaml = {
    "path": str(YOLO_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": names,
}

with DATA_YAML.open("w", encoding="utf-8") as file:
    yaml.safe_dump(data_yaml, file, sort_keys=False, allow_unicode=True)

print(DATA_YAML.read_text(encoding="utf-8")[:1000])

path: C:\training\segmentation_clean\yolo_dataset_clean_sem
train: images/train
val: images/val
names:
  0: candy
  1: egg tart
  2: french fries
  3: chocolate
  4: biscuit
  5: popcorn
  6: pudding
  7: ice cream
  8: cheese butter
  9: cake
  10: wine
  11: milkshake
  12: coffee
  13: juice
  14: milk
  15: tea
  16: almond
  17: red beans
  18: cashew
  19: dried cranberries
  20: soy
  21: walnut
  22: peanut
  23: egg
  24: apple
  25: date
  26: apricot
  27: avocado
  28: banana
  29: strawberry
  30: cherry
  31: blueberry
  32: raspberry
  33: mango
  34: olives
  35: peach
  36: lemon
  37: pear
  38: fig
  39: pineapple
  40: grape
  41: kiwi
  42: melon
  43: orange
  44: watermelon
  45: steak
  46: pork
  47: chicken duck
  48: sausage
  49: fried meat
  50: lamb
  51: sauce
  52: crab
  53: fish
  54: shellfish
  55: shrimp
  56: soup
  57: bread
  58: corn
  59: hamburg
  60: pizza
  61: hanamaki baozi
  62: wonton dumplings
  63: pasta
  64: noodles
  65: rice
  66: 

In [10]:
device = 0 if torch.cuda.is_available() else "cpu"
model_source = "yolo26s-sem.pt"

model = YOLO(model_source)

results = model.train(
    data=str(DATA_YAML),
    task="semantic",
    epochs=70,
    imgsz=512,
    batch=8,
    device=device,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=False,
    retina_masks=True,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3.0,
    weight_decay=0.0005,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.7,
    mixup=0.0,
    save_period=10, 
    cache="disk",
    patience=15,
    workers=6,
    amp=True,
    plots=True
)

Ultralytics 8.4.60  Python-3.14.2 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\training\segmentation_clean\food103_clean_sem.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-sem.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=yolo26l_seg_foodseg103-6, nbs=64, nms=False, opset=None, optim

In [ ]:
metrics = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    batch=16,
    device=device,
    split="val",
    project=str(PROJECT_DIR),
    name=f"{RUN_NAME}_val",
)
metrics

Ultralytics 8.4.60  Python-3.14.2 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
YOLO26s-sem summary (fused): 73 layers, 6,514,912 parameters, 0 gradients, 17.3 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 84.545.3 MB/s, size: 36.9 KB)
val: Scanning C:\training\segmentation_clean\yolo_dataset_clean_sem\labels\val.cache... 2121 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2121/2121 808.7Mit/s 0.0s
                 Class     Images     Pixels       mIoU     PixAcc: 100% ━━━━━━━━━━━━ 133/133 5.1it/s 26.1s<0.1s
                   all       2121  688572146      0.285       0.31
                 candy         11     509522      0.032     0.0354
              egg tart          1      44374          0          0
          french fries         76    5166347      0.445       0.61
             chocolate         18     985995       0.23      0.323
               biscuit         76    6103564      0.316      0.471
               popcorn          4     308435      

ultralytics.utils.metrics.SemanticMetrics object with attributes:

ap_class_index: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103]
cm_nc: 104
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000276CDF9EB30>
curves: []
curves_results: []
fitness: 0.28501367568969727
keys: ['metrics/mIoU', 'metrics/pixel_acc']
matrix: tensor([[1.8028e+04, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 1.6180e+03, 1.3638e+05],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00, 3.1320e+03],
        [5.3800e+02, 0.0000e+00, 3.1522e+06,  ..., 0.0000e+00, 2.0470e+03, 1.7725e+05],
        ...,
        [0.0000e